# Coachly NLU - Colab Drive Notebook (Qwen 0.5B)
Pipeline: mount drive -> setup files -> build dataset -> train QLoRA -> noisy STT eval.


In [3]:
# 1) Mount Drive + repo path
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/voice-ml-recognizer'
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Manca {REPO_DIR}.')
%cd {REPO_DIR}


Mounted at /content/drive
/content/drive/MyDrive/voice-ml-recognizer


In [9]:
# 2) GPU + deps
!nvidia-smi
!python -V

# Dipendenze minime stabili
!pip install -q -U "transformers>=4.44.0" "datasets>=2.20.0" "accelerate>=0.33.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" sentencepiece protobuf huggingface_hub


/bin/bash: line 1: nvidia-smi: command not found
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
googleapis-common-protos 1.73.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-aiplatform 1.141.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.

In [11]:
from pathlib import Path

p = Path("/content/drive/MyDrive/voice-ml-recognizer/colab_functiongemma_train.py")
s = p.read_text(encoding="utf-8")

# 1) import inspect (se manca)
if "import inspect" not in s:
    s = s.replace("import argparse\n", "import argparse\nimport inspect\n")

# 2) Patch Trainer init compatibility (tokenizer vs processing_class)
old = """    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["validation"],
        data_collator=collator,
        tokenizer=tokenizer,
    )
"""
new = """    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["validation"],
        data_collator=collator,
    )
    trainer_sig = inspect.signature(Trainer.__init__).parameters
    if "tokenizer" in trainer_sig:
        trainer_kwargs["tokenizer"] = tokenizer
    elif "processing_class" in trainer_sig:
        trainer_kwargs["processing_class"] = tokenizer
    trainer = Trainer(**trainer_kwargs)
"""
if old in s:
    s = s.replace(old, new)
else:
    print("Blocco Trainer già patchato o diverso dal template atteso.")

p.write_text(s, encoding="utf-8")
print("Patched:", p)

Patched: /content/drive/MyDrive/voice-ml-recognizer/colab_functiongemma_train.py


In [3]:
# 3) Ensure scripts in repo root (copy from refactor if missing)
import os, shutil

pairs = [
    ('dataset_creator.py', 'refactor/dataset_creator.py'),
    ('colab_functiongemma_train.py', 'refactor/colab_functiongemma_train.py'),
]
for dst, src in pairs:
    if not os.path.exists(dst):
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f'Copied {src} -> {dst}')
        else:
            raise FileNotFoundError(f'Mancano sia {dst} che {src}')
    else:
        print(f'OK: {dst}')


OK: dataset_creator.py
OK: colab_functiongemma_train.py


In [4]:
# 4) Compatibility patch for TrainingArguments (eval_strategy vs evaluation_strategy)
from pathlib import Path
import re

p = Path('colab_functiongemma_train.py')
s = p.read_text(encoding='utf-8')

# Rendi il file compatibile con le versioni che usano eval_strategy
if 'evaluation_strategy=' in s and 'inspect.signature(TrainingArguments.__init__)' not in s:
    s = s.replace('evaluation_strategy=', 'eval_strategy=')

p.write_text(s, encoding='utf-8')
print('Patched compatibility in', p)


Patched compatibility in colab_functiongemma_train.py


In [5]:
# 5) Build dataset
!python dataset_creator.py --output_dir data --per_action_per_lang 340 --unknown_per_lang 170

import json
from pathlib import Path
m = json.loads(Path('data/metadata.json').read_text(encoding='utf-8'))
print('Sizes:', m['sizes'])
print('Actions all:', m['stats']['all']['action'])


Dataset created in: data
{
  "all": 3060,
  "train": 2448,
  "val": 306,
  "test": 306
}
Action distribution (all): {'ADD_EXERCISE': 680, 'DELETE_EXERCISE': 680, 'LOG_SET': 680, 'UNKNOWN': 340, 'UPDATE_SET': 680}
Sizes: {'all': 3060, 'train': 2448, 'val': 306, 'test': 306}
Actions all: {'ADD_EXERCISE': 680, 'DELETE_EXERCISE': 680, 'LOG_SET': 680, 'UNKNOWN': 340, 'UPDATE_SET': 680}


In [12]:
# 6) Train QLoRA on Qwen 0.5B
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = 'output/functiongemma_qlora'

cmd = f"python colab_functiongemma_train.py --data_dir data --output_dir {OUTPUT_DIR} --base_model {BASE_MODEL}"
print(cmd)
!$cmd


python colab_functiongemma_train.py --data_dir data --output_dir output/functiongemma_qlora --base_model Qwen/Qwen2.5-0.5B-Instruct
Base model: Qwen/Qwen2.5-0.5B-Instruct
Loading dataset...
DatasetDict({
    train: Dataset({
        features: ['id', 'lang', 'action', 'text', 'label', 'messages'],
        num_rows: 2448
    })
    validation: Dataset({
        features: ['id', 'lang', 'action', 'text', 'label', 'messages'],
        num_rows: 306
    })
    test: Dataset({
        features: ['id', 'lang', 'action', 'text', 'label', 'messages'],
        num_rows: 306
    })
})
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:01<00:00, 181.96it/s]
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned ac

In [4]:
# 7) Quick metrics (generated by train script)
import json
from pathlib import Path
q = Path('output/functiongemma_qlora/quick_eval.json')
if q.exists():
    print(json.dumps(json.loads(q.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
else:
    print('quick_eval.json non trovato (training non concluso)')


quick_eval.json non trovato (training non concluso)


In [11]:
import json, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

ADAPTER_DIR = "output/functiongemma_qlora/adapter"
BASE_MODEL  = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SYSTEM = (
    "You are Coachly NLU. Convert workout speech-to-text into strict JSON.\nReturn ONLY valid JSON, no markdown.\nSchema:\n{\n  \"action\": \"ADD_EXERCISE|LOG_SET|UPDATE_SET|DELETE_EXERCISE|UNKNOWN\",\n  \"items\": [\n    {\n      \"exercise\": string,\n      \"sets\": integer|null,\n      \"reps\": integer|null,\n      \"weight\": number|null,\n      \"unit\": \"kg\"|\"lbs\"|null,\n      \"modifier\": \"to_failure\"|\"dropset\"|\"superset\"|\"amrap\"|\"pause\"|null\n    }\n  ]\n}"
)

TESTS = [
        # Multi-esercizio lungo
    "aggiungi bench press 3x10 80kg e squat 4x8 100kg e deadlift 5x5 140kg a cedimento",
    "add pull ups 4 sets to failure and dips 3x12 and bench press 3 sets of 8 at 80 kg dropset",

    # Ripensamenti in mezzo
    "aggiungi panca no aspetta squat 4x8 100kg e poi anche trazioni 3x6",
    "uhm fatto bench press no aspetta deadlift 5 reps 140 kg a cedimento",

    # Borderline: azione ambigua con blocchi misti log+add
    "fatto squat 10 reps 100kg e aggiungi panca piana 3x8",
    "done bench press 8 reps 80kg then add pull ups 4x6 to failure",

    # Solo esercizio + modifier, zero numeri
    "aggiungi panca superset trazioni a cedimento",

    # Tutto scritto male + multi blocco
    "agggiungi bencc pres 3x10 e squat quattro serie da otto e deaddlift cinq rip 140 chili dropset",

    # UNKNOWN mescolato con esercizi
    "quante calorie ho bruciato e poi aggiungi squat 4x8",
    "how many sets should i do and add bench press 3x10 80kg",
]

print(f"Loading on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
base  = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32, device_map=DEVICE)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()
print("Pronto.\n")

def predict(text):
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}],
        tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=150, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw)
    try:    return json.JSONDecoder().raw_decode(raw)[0]
    except: m = re.search(r'\{.*\}', raw, re.DOTALL); return json.loads(m.group(0)) if m else {"_raw": raw}

for text in TESTS:
    result = predict(text)
    action = result.get("action", "?")
    print(f"[{action:16s}] {text}")
    print(f"  {json.dumps(result, ensure_ascii=False)}\n")

Loading on cpu...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Pronto.

[ADD_EXERCISE    ] aggiungi bench press 3x10 80kg e squat 4x8 100kg e deadlift 5x5 140kg a cedimento
  {"action": "ADD_EXERCISE", "items": [{"exercise": "bench press", "sets": 3, "reps": 10, "weight": 80.0, "unit": "kg", "modifier": null}, {"exercise": "squat", "sets": 4, "reps": 8, "weight": 100.0, "unit": "kg", "modifier": null}, {"exercise": "deadlift", "sets": 5, "reps": 5, "weight": 140.0, "unit": "kg", "modifier": "to_failure"}]}

[ADD_EXERCISE    ] add pull ups 4 sets to failure and dips 3x12 and bench press 3 sets of 8 at 80 kg dropset
  {"action": "ADD_EXERCISE", "items": [{"exercise": "pull up", "sets": 4, "reps": null, "weight": null, "unit": null, "modifier": "to_failure"}, {"exercise": "dumbbell dip", "sets": 3, "reps": 12, "weight": null, "unit": null, "modifier": null}, {"exercise": "bench press", "sets": 3, "reps": 8, "weight": 80.0, "unit": "kg", "modifier": "dropset"}]}

[ADD_EXERCISE    ] aggiungi panca no aspetta squat 4x8 100kg e poi anche trazioni 3x6
  {